In [2]:
import os
import joblib
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
import psycopg2

def get_db_connection():
    return psycopg2.connect(
        dbname=os.getenv("DB_NAME", "smartlogix_db"),
        user=os.getenv("DB_USER", "postgres"),
        password=os.getenv("DB_PASSWORD", "password"),
        host=os.getenv("DB_HOST", "localhost"),
        port=os.getenv("DB_PORT", "5432")
    )

def train_optimized_model():
    print("🔄 Fetching advanced fleet & maintenance data from PostgreSQL...")
    conn = get_db_connection()
    
    # Advanced query joining fleet with historical maintenance aggregates
    query = """
        SELECT 
            f.vehicle_id, 
            f.vehicle_type, 
            f.capacity_kg, 
            f.max_range_km, 
            f.avg_speed_kmph, 
            f.odometer_km, 
            f.fleet_status, 
            f.battery_capacity_wh,
            COUNT(m.work_order_id) as total_service_count,
            COALESCE(SUM(m.downtime_hours), 0) as total_downtime_hours,
            COALESCE(SUM(m.cost_inr), 0) as total_maintenance_cost
        FROM fleet_vehicles f
        LEFT JOIN maintenance_history m ON f.vehicle_id = m.vehicle_id
        GROUP BY f.vehicle_id, f.vehicle_type, f.capacity_kg, f.max_range_km, 
                 f.avg_speed_kmph, f.odometer_km, f.fleet_status, f.battery_capacity_wh
    """
    df = pd.read_sql(query, conn)
    
    if df.empty:
        print("❌ Error: No records found.")
        conn.close()
        return

    # Target variable definition
    df["needs_maintenance"] = df.apply(lambda row: 1 if str(row["fleet_status"]).strip().lower() == "maintenance" or float(row["odometer_km"] or 0) > 60000 or row["total_downtime_hours"] > 30 else 0, axis=1)

    # Expanded feature set including history metrics
    feature_cols = [
        "capacity_kg", 
        "max_range_km", 
        "avg_speed_kmph", 
        "odometer_km", 
        "battery_capacity_wh",
        "total_service_count",
        "total_downtime_hours",
        "total_maintenance_cost"
    ]
    
    X = df[feature_cols].fillna(0)
    y = df["needs_maintenance"]

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

    print("🚀 Training Optimized Random Forest Classifier (with class balancing)...")
    # class_weight='balanced' helps the model pay equal attention to the minority class
    model = RandomForestClassifier(
        n_estimators=150, 
        max_depth=12, 
        min_samples_split=4, 
        class_weight='balanced', 
        random_state=42
    )
    model.fit(X_train, y_train)

    joblib.dump(model, "fleet_maintenance_model.pkl")

    # Evaluation
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n📈 OPTIMIZED MODEL ACCURACY: {acc * 100:.2f}%\n")
    print(classification_report(y_test, y_pred, target_names=["Healthy (0)", "Maintenance Req (1)"]))
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))

    # Save back to database
    print("\n✍️ Updating database with optimized predictions...")
    df["predicted_maintenance"] = model.predict(X)
    
    cursor = conn.cursor()
    update_query = "UPDATE fleet_vehicles SET predicted_maintenance = %s WHERE vehicle_id = %s"
    update_data = [(int(row["predicted_maintenance"]), row["vehicle_id"]) for _, row in df.iterrows()]
    cursor.executemany(update_query, update_data)
    
    conn.commit()
    cursor.close()
    conn.close()
    print("✅ Success! Database updated with optimized model output.")

if __name__ == "__main__":
    train_optimized_model()

🔄 Fetching advanced fleet & maintenance data from PostgreSQL...
🚀 Training Optimized Random Forest Classifier (with class balancing)...


C:\Users\VIDHYA\AppData\Local\Temp\ipykernel_6924\3528880757.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)



📈 OPTIMIZED MODEL ACCURACY: 99.32%

                     precision    recall  f1-score   support

        Healthy (0)       0.89      1.00      0.94         8
Maintenance Req (1)       1.00      0.99      1.00       140

           accuracy                           0.99       148
          macro avg       0.94      1.00      0.97       148
       weighted avg       0.99      0.99      0.99       148

Confusion Matrix:
[[  8   0]
 [  1 139]]

✍️ Updating database with optimized predictions...
✅ Success! Database updated with optimized model output.
